In [1]:
### misc
import pandas as pd
import numpy as np
import os
from pathlib import Path
import pickle
import time
from itertools import product

#### graphical
import matplotlib.pyplot as plt
import corner

#### ML
import sklearn
from sklearn.decomposition import PCA
import tensorflow as tf
import keras
from keras import layers

from WMSE import WMSE, WMSE_metric

##### poke gpu
os.environ["CUDA_VISIBLE_DEVICES"]="1"

physical_devices = tf.config.list_physical_devices("GPU") 

tf.config.experimental.set_memory_growth(physical_devices[0], True)

gpu0usage = tf.config.experimental.get_memory_info("GPU:0")["current"]

print("Current GPU usage:\n"
     + " - GPU0: " + str(gpu0usage) + "B\n")

modelpath = '/home/hatte/M4/models'

2025-04-03 15:32:35.206862: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1743690755.219114 3941276 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1743690755.222546 3941276 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-04-03 15:32:35.235606: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Current GPU usage:
 - GPU0: 0B



I0000 00:00:1743690756.641138 3941276 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 14957 MB memory:  -> device: 0, name: NVIDIA RTX A4500, pci bus id: 0000:61:00.0, compute capability: 8.6


In [2]:
def scheduler(epoch, lr,):
    ## Learning rate scheduler
    # Decreases learning rate in-training for stability
    if lr < 1e-5:
        return float(lr)
    else:
        return float(lr * tf.math.exp(-1e-4))

In [3]:
df_full = pd.read_hdf('../grids/Chiara.hdf5', key='df') ## edit for your grid!!

df_full['logLPhot'] = np.log10(df_full['LPhot'])

df_full['lognumax'] = np.log10(df_full['numax']*3090)

df_full['logdnuSer'] = np.log10(df_full['dnuSer']*135)

#### define inputs
inputs = ['massini', 'zini', 'yini', 'alphaMLT', 'age', 'eta', 'alphaFe']

#### define outputs
classical_outputs = ['FeH', 'logLPhot', 'Teff']
astero_outputs = ['numax', 'dnuSer'] 

outputs = classical_outputs+astero_outputs

df = df_full[inputs+outputs]

df_norm = (df - df.min())/(df.max() - df.min())

## check df_norm.describe looks reasonable (min=0, max=1):
df_norm.describe()

#### train/test split with seed 
seed = 42

df_train = df_norm.sample(frac=0.95, random_state=seed)
df_test = df_norm.drop(df_train.index)

df_train_inputs, df_val_inputs, df_train_outputs, df_val_outputs = sklearn.model_selection.train_test_split(df_train[inputs],df_train[outputs], test_size = 0.05, random_state=seed)

print("Training set: ", len(df_train_inputs))
print("Validation set: ", len(df_val_inputs))
print("Test set: ", len(df_test))

Training set:  6754081
Validation set:  355478
Test set:  374187


In [4]:
#unnormed_weights_dict = {'FeH':0.01, 'logLPhot':0.001, 'Teff':10, 'numax':0.001/3090, 'dnuSer':0.0001/135}

unnormed_weights_dict = {'FeH':0.01, 'logLPhot':0.001, 'Teff':10, 'numax':0.01/3090, 'dnuSer':0.01/135}

unnormed_weights = list(unnormed_weights_dict.values())

weights = [2*unnormed_weights_dict[i]/(df[i].max() - df[i].min()) for i in outputs]

In [5]:
n_dense_layers = 6

dense_layer_units = 128

Nepochs = 20000

learning_rate = 0.00005

model_name = 'smart-logLPhot-numax-dnuSer-exponent-1e-4'

loss_func = 'WMSE'

df_train_inputs.join(df_train_outputs).to_hdf(f'{modelpath}/long-runs/training-data/training-{model_name}-nlayers-{n_dense_layers}-nunits-{dense_layer_units}-epochs-{Nepochs}-lrate-{learning_rate}-lossfunc-{loss_func}.hdf5', key = 'df')

In [6]:
checkpoint_dir = f'{modelpath}/long-runs/checkpoint/chk-{model_name}-nlayers-{n_dense_layers}-nunits-{dense_layer_units}-epochs-{Nepochs}-lrate-{learning_rate}-lossfunc-{loss_func}.model.keras'

full_model_dir = f'{modelpath}/long-runs/full-model/mod-{model_name}-nlayers-{n_dense_layers}-nunits-{dense_layer_units}-epochs-{Nepochs}-lrate-{learning_rate}-lossfunc-{loss_func}/'

if not os.path.exists(full_model_dir):
    os.makedirs(full_model_dir)

historyfile = f'{modelpath}/long-runs/history/hist-{model_name}-nlayers-{n_dense_layers}-nunits-{dense_layer_units}-epochs-{Nepochs}-lrate-{learning_rate}-lossfunc-{loss_func}.json'
    
cp_callback = tf.keras.callbacks.ModelCheckpoint(filepath = checkpoint_dir, verbose = 1, save_best_only = True, save_freq = 'epoch')

lr_callback = tf.keras.callbacks.LearningRateScheduler(scheduler, )

In [ ]:
######## map out model architecture
#### input layer
nn_input = keras.Input(shape=(len(inputs),))

#### dense layer(s)
for n_dense_layer in range(n_dense_layers):
    if n_dense_layer == 0:
        dense_layer = layers.Dense(dense_layer_units, activation='relu')(nn_input)
    else:
        dense_layer = layers.Dense(dense_layer_units, activation='relu')(dense_layer)

#### output layer
nn_output =  layers.Dense(len(outputs), activation='linear')(dense_layer)

######## store architecture as keras model
model = keras.Model(inputs=nn_input, outputs=nn_output, name=model_name)

tb_callback = tf.keras.callbacks.TensorBoard(log_dir = f'{modelpath}/logs/long-runs/log-{model_name}-nlayers-{n_dense_layers}-nunits-{dense_layer_units}-epochs-{Nepochs}-lrate-{learning_rate}-lossfunc-{loss_func}')

model.compile(loss=WMSE(weights), optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate))

history = model.fit(df_train_inputs,
          df_train_outputs,
          validation_data=(df_val_inputs,df_val_outputs),
          batch_size=2**16, #change higher
          verbose=1,
          epochs=Nepochs,
          shuffle=True, callbacks = [tb_callback, cp_callback, lr_callback]) 

tf.saved_model.save(model, full_model_dir)
hist_df = pd.DataFrame(history.history)
    
with open(os.path.join(modelpath, historyfile), mode="w") as f:
    hist_df.to_json(f)

Epoch 1/20000


I0000 00:00:1743690791.117773 3942729 service.cc:148] XLA service 0x73e5cc00e4e0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1743690791.117798 3942729 service.cc:156]   StreamExecutor device (0): NVIDIA RTX A4500, Compute Capability 8.6
2025-04-03 15:33:11.144851: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1743690791.271272 3942729 cuda_dnn.cc:529] Loaded cuDNN version 90300
2025-04-03 15:33:11.338715: W external/local_xla/xla/service/gpu/nvptx_compiler.cc:930] The NVIDIA driver's CUDA version is 12.2 which is older than the PTX compiler version 12.5.82. Because the driver is older than the PTX compiler version, XLA is disabling parallel compilation, which may slow down compilation. You should update your NVIDIA driver or use the NVIDIA-provided CUDA forward compatibility packages.
2025-04-03 15:33:11.91581

 27/104 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 6560150.5000

I0000 00:00:1743690793.479815 3942729 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


102/104 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 5894696.0000

2025-04-03 15:33:14.845359: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_258', 32 bytes spill stores, 32 bytes spill loads

2025-04-03 15:33:14.953867: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_451', 24 bytes spill stores, 24 bytes spill loads

2025-04-03 15:33:14.978906: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_451_0', 36 bytes spill stores, 36 bytes spill loads

2025-04-03 15:33:15.018258: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_461', 24 bytes spill stores, 48 bytes spill loads

2025-04-03 15:33:15.139568: I external/local_xla/xla/stream_ex

104/104 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - loss: 5887184.0000

2025-04-03 15:33:17.387652: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_30', 32 bytes spill stores, 32 bytes spill loads

2025-04-03 15:33:17.514616: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_30', 616 bytes spill stores, 440 bytes spill loads




Epoch 1: val_loss improved from inf to 5288030.50000, saving model to /home/hatte/M4/models/long-runs/checkpoint/chk-smart-logLPhot-numax-dnuSer-exponent-1e-4-nlayers-6-nunits-128-epochs-20000-lrate-5e-05-lossfunc-WMSE.model.keras
104/104 ━━━━━━━━━━━━━━━━━━━━ 8s 42ms/step - loss: 5883534.5000 - val_loss: 5288030.5000 - learning_rate: 4.9995e-05
Epoch 2/20000
101/104 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 5284743.0000
Epoch 2: val_loss improved from 5288030.50000 to 5238717.00000, saving model to /home/hatte/M4/models/long-runs/checkpoint/chk-smart-logLPhot-numax-dnuSer-exponent-1e-4-nlayers-6-nunits-128-epochs-20000-lrate-5e-05-lossfunc-WMSE.model.keras
104/104 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 5284070.0000 - val_loss: 5238717.0000 - learning_rate: 4.9990e-05
Epoch 3/20000
103/104 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 5233431.5000
Epoch 3: val_loss improved from 5238717.00000 to 5197861.00000, saving model to /home/hatte/M4/models/long-runs/checkpoint/chk-smart-logLPhot-nu